# IMF WEO Data Extraction

**Purpose:** Reshape the IMF World Economic Outlook Excel file from wide-by-year format  
into panel format suitable for the sovereign risk pipeline.

**Input:** `data/raw/world_economic_outlook_imf.xls`  
**Output:** `data/raw/weo_imf.csv`

**Source format (wide):**  
Each row = one country × one WEO variable. Year columns span 1980–2024.

| ISO | WEO Subject Code | Subject Descriptor | 1980 | 1981 | … | 2024 |
|-----|------------------|--------------------|------|------|---|------|
| USA | NGDP_R           | GDP, constant prices | 5123 | 5287 | … | 9876 |

**Target format (panel):**  
Each row = one country × one year. Each WEO variable becomes a column.

| iso | year | GDP, constant prices | Inflation, average … | … |
|-----|------|----------------------|----------------------|---|
| USA | 1980 | 5123                 | 13.5                 | … |

## Cell 1 — Imports and project root

In [1]:
import sys
import logging
from pathlib import Path

import pandas as pd
import numpy as np


def find_project_root(start: Path = Path().resolve()) -> Path:
    """Walk up directory tree until the config/ folder is found."""
    for directory in [start, *start.parents]:
        if (directory / "config").is_dir():
            return directory
    raise FileNotFoundError(
        "Cannot find project root. "
        "Expected a config/ folder somewhere above this notebook."
    )


PROJECT_ROOT = find_project_root(Path("__file__").resolve().parent if "__file__" in dir() else Path().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")

Project root : C:\Users\nduka\OneDrive\Desktop\Claude\sovereign-risk-agent
Python       : 3.14.3


## Cell 2 — Config and logging

In [5]:
from config.settings import RAW_DATA_DIR, LOG_FORMAT, LOG_DATE_FORMAT

logging.basicConfig(
    level=logging.INFO,
    format=LOG_FORMAT,
    datefmt=LOG_DATE_FORMAT,
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("weo_extraction")

INPUT_FILE  = RAW_DATA_DIR / "world_economic_outlook_imf.xlsx"
OUTPUT_FILE = RAW_DATA_DIR / "weo_imf.csv"

log.info("Input  : %s", INPUT_FILE)
log.info("Output : %s", OUTPUT_FILE)
log.info("Exists : %s", INPUT_FILE.exists())

2026-04-07 18:57:23 | INFO     | weo_extraction | Input  : C:\Users\nduka\OneDrive\Desktop\Claude\sovereign-risk-agent\data\raw\world_economic_outlook_imf.xlsx
2026-04-07 18:57:23 | INFO     | weo_extraction | Output : C:\Users\nduka\OneDrive\Desktop\Claude\sovereign-risk-agent\data\raw\weo_imf.csv
2026-04-07 18:57:23 | INFO     | weo_extraction | Exists : True


## Cell 3 — Load the WEO Excel file

The IMF WEO workbook typically contains one data sheet.  
We load it, inspect its shape, and identify the key structural columns.

In [7]:
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)

   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   --------------------

In [9]:
# Inspect available sheets first
xf = pd.ExcelFile(INPUT_FILE, engine="openpyxl")
log.info("Sheets: %s", xf.sheet_names)

raw = pd.read_excel(
    INPUT_FILE,
    sheet_name=0,
    engine="openpyxl",
    dtype=str,          # load everything as strings; we parse numerics later
    na_values=["", "n/a", "--", "...", "NA"],
)

log.info("Loaded  : %d rows × %d columns", *raw.shape)
print("\nFirst 5 rows (first 10 columns):")
print(raw.iloc[:5, :10].to_string())
print("\nAll column names:")
print(raw.columns.tolist())

2026-04-07 19:00:16 | INFO     | weo_extraction | Sheets: ['world_economic_outlook_imf']


2026-04-07 19:00:17 | INFO     | weo_extraction | Loaded  : 8626 rows × 60 columns

First 5 rows (first 10 columns):
  WEO Country Code  ISO WEO Subject Code      Country                       Subject Descriptor                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

## Cell 4 — Identify structural vs year columns

WEO files mix metadata columns (ISO code, Subject Descriptor, Units, etc.)  
with numeric year columns (1980 … 2024). We split these two groups here.

In [10]:
# Key identifier columns — these names appear in every WEO release.
# Adjust the mapping below if your file uses slightly different header text.
COL_MAP = {
    "ISO":                "iso",
    "WEO Subject Code":   "weo_subject_code",
    "Subject Descriptor": "subject_descriptor",
    "Units":              "units",
    "Scale":              "scale",
}

# Verify all expected columns exist
missing_cols = [c for c in COL_MAP if c not in raw.columns]
if missing_cols:
    log.warning("These expected columns are NOT in the file: %s", missing_cols)
    log.warning("Available columns: %s", raw.columns.tolist())
else:
    log.info("All key identifier columns found.")

# Identify year columns: columns whose names are 4-digit integers in 1900-2100
def is_year_col(col_name: str) -> bool:
    try:
        yr = int(str(col_name).strip())
        return 1900 <= yr <= 2100
    except (ValueError, TypeError):
        return False

year_cols = [c for c in raw.columns if is_year_col(c)]
meta_cols = [c for c in raw.columns if not is_year_col(c)]

log.info("Year columns : %d  (%s … %s)", len(year_cols), year_cols[0], year_cols[-1])
log.info("Meta columns : %d  %s", len(meta_cols), meta_cols)

2026-04-07 19:01:43 | INFO     | weo_extraction | All key identifier columns found.
2026-04-07 19:01:43 | INFO     | weo_extraction | Year columns : 50  (1980 … 2029)
2026-04-07 19:01:43 | INFO     | weo_extraction | Meta columns : 10  ['WEO Country Code', 'ISO', 'WEO Subject Code', 'Country', 'Subject Descriptor', 'Subject Notes', 'Units', 'Scale', 'Country/Series-specific Notes', 'Estimates Start After']


## Cell 5 — Drop trailing metadata and clean values

The WEO workbook often has footer rows (source notes, disclaimers) and an  
`Estimates Start After` column that we don't need for the panel. We drop those  
and clean numeric values (commas in thousands, stray whitespace).

In [11]:
# Keep only rows that have a valid ISO code (drops footer/note rows)
df = raw.copy()
df = df[df["ISO"].notna()].copy()

# Drop housekeeping columns we don't need in the panel
DROP_COLS = [
    "WEO Country Code",
    "Subject Notes",
    "Country/Series-specific Notes",
    "Estimates Start After",
]
df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# Rename identifier columns to snake_case
df = df.rename(columns={k: v for k, v in COL_MAP.items() if k in df.columns})

# Clean numeric year values: strip whitespace, remove thousands commas
for col in year_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(",", "", regex=False)  # "1,234.5" -> "1234.5"
        .replace({"nan": np.nan, "n/a": np.nan, "--": np.nan, "...": np.nan, "": np.nan})
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

log.info("After cleaning: %d rows × %d columns", *df.shape)
log.info("Unique countries (ISO): %d", df["iso"].nunique())
log.info("Unique variables      : %d", df["weo_subject_code"].nunique())
print("\nSample rows:")
print(df[["iso", "weo_subject_code", "subject_descriptor"] + year_cols[:5]].head(8).to_string(index=False))

2026-04-07 19:02:57 | INFO     | weo_extraction | After cleaning: 8624 rows × 56 columns
2026-04-07 19:02:57 | INFO     | weo_extraction | Unique countries (ISO): 196
2026-04-07 19:02:57 | INFO     | weo_extraction | Unique variables      : 44

Sample rows:
iso weo_subject_code                                 subject_descriptor  1980  1981  1982  1983  1984
AFG           NGDP_R            Gross domestic product, constant prices   NaN   NaN   NaN   NaN   NaN
AFG        NGDP_RPCH            Gross domestic product, constant prices   NaN   NaN   NaN   NaN   NaN
AFG             NGDP             Gross domestic product, current prices   NaN   NaN   NaN   NaN   NaN
AFG            NGDPD             Gross domestic product, current prices   NaN   NaN   NaN   NaN   NaN
AFG           PPPGDP             Gross domestic product, current prices   NaN   NaN   NaN   NaN   NaN
AFG           NGDP_D                   Gross domestic product, deflator   NaN   NaN   NaN   NaN   NaN
AFG          NGDPRPC Gross d

## Cell 6 — Reshape: wide → long → panel

**Step 1 (melt):** Unpivot year columns → one row per (ISO, variable, year).  
**Step 2 (pivot_table):** Pivot Subject Descriptor → columns, giving one row per (ISO, year).  

We use `subject_descriptor` as column names because they are human-readable.  
The `weo_subject_code` → `subject_descriptor` mapping is printed at the end for reference.

In [12]:
# ── Step 1: melt year columns to long format ──────────────────────────────────
id_vars = [c for c in ["iso", "weo_subject_code", "subject_descriptor", "units", "scale"]
           if c in df.columns]

long = df.melt(
    id_vars=id_vars,
    value_vars=year_cols,
    var_name="year",
    value_name="value",
)
long["year"] = long["year"].astype(int)

log.info("Long format  : %d rows × %d columns", *long.shape)

# ── Step 2: pivot subject_descriptor → columns ────────────────────────────────
# aggfunc='first' handles the rare case of duplicate (iso, year, subject) rows
panel = long.pivot_table(
    index=["iso", "year"],
    columns="subject_descriptor",
    values="value",
    aggfunc="first",
).reset_index()

# Flatten the MultiIndex column that pivot_table produces
panel.columns.name = None

# Sort by country then year
panel = panel.sort_values(["iso", "year"]).reset_index(drop=True)

log.info("Panel format : %d rows × %d columns", *panel.shape)
log.info("Countries    : %d", panel["iso"].nunique())
log.info("Years        : %d → %d", panel["year"].min(), panel["year"].max())
print("\nPanel columns:")
for i, col in enumerate(panel.columns):
    print(f"  [{i:3d}] {col}")

2026-04-07 19:03:11 | INFO     | weo_extraction | Long format  : 431200 rows × 7 columns
2026-04-07 19:03:11 | INFO     | weo_extraction | Panel format : 9059 rows × 30 columns
2026-04-07 19:03:11 | INFO     | weo_extraction | Countries    : 196
2026-04-07 19:03:11 | INFO     | weo_extraction | Years        : 1980 → 2029

Panel columns:
  [  0] iso
  [  1] year
  [  2] Current account balance
  [  3] Employment
  [  4] General government gross debt
  [  5] General government net debt
  [  6] General government net lending/borrowing
  [  7] General government primary net lending/borrowing
  [  8] General government revenue
  [  9] General government structural balance
  [ 10] General government total expenditure
  [ 11] Gross domestic product based on purchasing-power-parity (PPP) share of world total
  [ 12] Gross domestic product corresponding to fiscal year, current prices
  [ 13] Gross domestic product per capita, constant prices
  [ 14] Gross domestic product per capita, current pr

## Cell 7 — WEO subject code reference table

Print the mapping between `WEO Subject Code` and `Subject Descriptor`  
so it's easy to look up which code corresponds to which column.

In [13]:
ref_cols = [c for c in ["weo_subject_code", "subject_descriptor", "units", "scale"] if c in df.columns]
reference = (
    df[ref_cols]
    .drop_duplicates(subset=["weo_subject_code"])
    .sort_values("weo_subject_code")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
print(f"Total WEO variables: {len(reference)}\n")
print(reference.to_string(index=False))

Total WEO variables: 44

weo_subject_code                                                                 subject_descriptor                                              units    scale
             BCA                                                            Current account balance                                       U.S. dollars Billions
       BCA_NGDPD                                                            Current account balance                                     Percent of GDP    Units
             GGR                                                         General government revenue                                  National currency Billions
        GGR_NGDP                                                         General government revenue                                     Percent of GDP    Units
            GGSB                                              General government structural balance                                  National currency Billions
      GGSB_NPGD

## Cell 8 — Data quality check

Completeness by column — flagged as OK / SPARSE / VERY SPARSE  
using the same thresholds as the World Bank notebook.

In [14]:
variable_cols = [c for c in panel.columns if c not in ("iso", "year")]
total_rows    = len(panel)

print(f"{'Column':<55} {'Non-null':>9} {'Complete':>9}  Status")
print("-" * 90)

for col in variable_cols:
    non_null     = panel[col].notna().sum()
    completeness = non_null / total_rows * 100
    if completeness >= 80:
        status = "OK"
    elif completeness >= 50:
        status = "SPARSE"
    else:
        status = "VERY SPARSE"
    print(f"{col:<55} {non_null:>9,} {completeness:>8.1f}%  {status}")

print("-" * 90)
print(f"{'Total rows':<55} {total_rows:>9,}")

Column                                                   Non-null  Complete  Status
------------------------------------------------------------------------------------------
Current account balance                                     8,610     95.0%  OK
Employment                                                  1,684     18.6%  VERY SPARSE
General government gross debt                               6,693     73.9%  SPARSE
General government net debt                                 3,014     33.3%  VERY SPARSE
General government net lending/borrowing                    7,376     81.4%  OK
General government primary net lending/borrowing            6,924     76.4%  SPARSE
General government revenue                                  7,451     82.2%  OK
General government structural balance                       3,036     33.5%  VERY SPARSE
General government total expenditure                        7,406     81.8%  OK
Gross domestic product based on purchasing-power-parity (PPP) share of

## Cell 9 — Preview sample countries

In [15]:
preview_countries = ["USA", "BRA", "NGA"]
preview_years     = range(2018, 2025)   # last 6 years

mask = panel["iso"].isin(preview_countries) & panel["year"].isin(preview_years)
preview_cols = ["iso", "year"] + variable_cols[:6]   # first 6 variables

pd.set_option("display.max_columns", None)
pd.set_option("display.width",       200)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

print(panel.loc[mask, preview_cols].sort_values(["iso", "year"]).to_string(index=False))

iso  year  Current account balance  Employment  General government gross debt  General government net debt  General government net lending/borrowing  General government primary net lending/borrowing
BRA  2018                  -53.818         NaN                       5937.900                     3695.840                                  -489.599                                           -60.719
BRA  2019                  -65.001         NaN                       6437.300                     4041.770                                  -359.003                                            -6.116
BRA  2020                  -24.914         NaN                       7305.730                     4670.000                                  -885.520                                          -573.449
BRA  2021                  -40.409         NaN                       8014.880                     4966.920                                  -237.087                                           178.308
BRA  

## Cell 10 — Save to CSV

In [16]:
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUTPUT_FILE, index=False)

size_kb = OUTPUT_FILE.stat().st_size / 1024

log.info("Saved      : %s", OUTPUT_FILE)
log.info("Rows       : %d", len(panel))
log.info("Columns    : %d  (iso, year + %d variables)", len(panel.columns), len(variable_cols))
log.info("File size  : %.1f KB", size_kb)
log.info("Countries  : %d  |  Years: %d–%d",
         panel["iso"].nunique(),
         panel["year"].min(),
         panel["year"].max())
log.info("")
log.info("Next steps:")
log.info("  1. Join weo_imf.csv with world_bank_data.csv on (iso / country, year).")
log.info("  2. Select the WEO variables relevant to the sovereign risk model.")
log.info("  3. Populate COUNTRIES_BY_TYPE in config/settings.py and run extract.ipynb.")

2026-04-07 19:04:43 | INFO     | weo_extraction | Saved      : C:\Users\nduka\OneDrive\Desktop\Claude\sovereign-risk-agent\data\raw\weo_imf.csv
2026-04-07 19:04:43 | INFO     | weo_extraction | Rows       : 9059
2026-04-07 19:04:43 | INFO     | weo_extraction | Columns    : 30  (iso, year + 28 variables)
2026-04-07 19:04:43 | INFO     | weo_extraction | File size  : 1540.7 KB
2026-04-07 19:04:43 | INFO     | weo_extraction | Countries  : 196  |  Years: 1980–2029
2026-04-07 19:04:43 | INFO     | weo_extraction | 
2026-04-07 19:04:43 | INFO     | weo_extraction | Next steps:
2026-04-07 19:04:43 | INFO     | weo_extraction |   1. Join weo_imf.csv with world_bank_data.csv on (iso / country, year).
2026-04-07 19:04:43 | INFO     | weo_extraction |   2. Select the WEO variables relevant to the sovereign risk model.
2026-04-07 19:04:43 | INFO     | weo_extraction |   3. Populate COUNTRIES_BY_TYPE in config/settings.py and run extract.ipynb.
